In [ ]:
# ============================================
# MEDITRIAGE AI - JUPYTER NOTEBOOK
# Contains draw_mermaid_png() graph visualization
# Run all cells to see the graph
# ============================================

# Cell 1: Install and Import
!pip install langgraph langchain langchain-openai pydantic python-dotenv

import os
from typing import TypedDict, Annotated, List
from langgraph.graph import StateGraph, END
from langgraph.checkpoint import MemorySaver
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from IPython.display import Image, display

# Cell 2: Define State (TypedDict requirement)
class AgentState(TypedDict):
    messages: List
    symptoms: str
    questions_asked: List[str]
    confidence_score: float
    triage_level: str
    recommendations: List[str]
    iteration_count: int
    assessment_complete: bool

# Cell 3: Define Structured Output (Pydantic requirement)
class SymptomAssessment(BaseModel):
    symptom_category: str
    confidence_score: float
    recommended_action: str

# Cell 4: Define Nodes
def symptom_analyzer_node(state: AgentState) -> AgentState:
    return {**state, "confidence_score": 0.8, "iteration_count": state.get("iteration_count", 0) + 1}

def clarification_node(state: AgentState) -> AgentState:
    questions = state.get("questions_asked", [])
    return {**state, "questions_asked": questions + ["How long have you had these symptoms?"]}

def triage_router_node(state: AgentState) -> AgentState:
    return {**state, "triage_level": "home_care", "assessment_complete": True}

def resource_generator_node(state: AgentState) -> AgentState:
    return {**state, "recommendations": ["Rest and stay hydrated"]}

# Cell 5: Conditional Edge Function
def should_continue_clarifying(state: AgentState) -> str:
    if state["confidence_score"] < 0.7 and state.get("iteration_count", 0) < 3:
        return "clarify"
    return "route"

# Cell 6: Build Graph
workflow = StateGraph(AgentState)
workflow.add_node("symptom_analyzer", symptom_analyzer_node)
workflow.add_node("clarification", clarification_node)
workflow.add_node("triage_router", triage_router_node)
workflow.add_node("resource_generator", resource_generator_node)

workflow.set_entry_point("symptom_analyzer")
workflow.add_conditional_edges("symptom_analyzer", should_continue_clarifying, {"clarify": "clarification", "route": "triage_router"})
workflow.add_edge("clarification", "symptom_analyzer")
workflow.add_edge("triage_router", "resource_generator")
workflow.add_edge("resource_generator", END)

graph = workflow.compile()

# Cell 7: DRAW GRAPH VISUALIZATION (REQUIRED for assignment)
# This generates the mermaid PNG image
display(Image(graph.get_graph().draw_mermaid_png()))

# Cell 8: Test the graph
initial_state = {
    "messages": [],
    "symptoms": "I have a headache",
    "questions_asked": [],
    "confidence_score": 0.0,
    "triage_level": "",
    "recommendations": [],
    "iteration_count": 0,
    "assessment_complete": False
}

result = graph.invoke(initial_state)
print("Final State:", result)